### Step 1: Import Libraries & API Key

In [1]:
import os #To interact with the operating system and access environment variables
from openai import OpenAI #To interact with the OpenAI API
from dotenv import load_dotenv #To load environment variables from the .env file
from IPython.display import Markdown #So we can print in markdown format in Jupyter Notebook

load_dotenv() #Load environment variables from .env file

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") #Get the API key from environment variables

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY is missing.") #Checks if the API key is present in the environment file. If not, it raises an exception (Error message). If no error message, we are good.

Gradio is a tool developed by Hugging Face. Allows us to easily create different types of user interfaces (it's a UI tool). https://gradio.app/
We will use the chatbot interface. https://gradio.app/main/docs/gradio/chatbot


The chatbot interface is specifically desgined for you to work with LLMs (geared towards that).

### Step 2: Basic UI Interface

In [2]:
import gradio as gr

#Getting an warning here is normal

In [3]:
def response_basic(message, history): # We define a function (needed to use for Gradio) that takes a message and a history as input (so 2 paramaters). The history is a list of previous messages and responses.
    return "I'm crazy about AI!" #So the function will not use those parameters (message and history) and just return this

In [4]:
#Creating User Interface with Gradio:
gr.ChatInterface(fn=response_basic).launch()
#We create a chat interface using Gradio's ChatInterface class. We pass the response_basic function as the fn parameter, which will be called whenever the user sends a message.
# fn is the function we want to pass in the ChatInterface. The function will be called whenever the user sends a message, and it will return a response that will be displayed in the chat interface.

# We're calling the ChatInterface's launch method to start the interface. This will open a new window in the browser where users can interact with the chatbot.
# We're calling the ChatInterface tool, and then takes the function (response_basic) and the message as an argument. The function will be called whenever the user sends a message,
# and it will return a response that will be displayed in the chat interface.

# So for example, if the user types "Hello!" as the message, it will go into this user interface and go into the message parameter of the response_basic function,
# and then the function will return "I'm crazy about AI!" as the response (doesnt use the message or history parameters at all as we have defined the function), which will be displayed in the chat interface.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


So the chatbot is always responding with "I'm crazy about AI!" no matter what the user inputs because that is the way we defined our function.

First, the message goes into our UI. Then the response_basic function is called with that message as an input and its previous messages (history) and it does nothing with it. It returns "I'm crazy about AI!" regardless of the message or history, since that's how we defined our function.

#### Let's look at the chat history:

In [5]:
def response_basic(message, history): # We define a function (needed to use Gradio) that takes a message and a history as input (so 2 paramaters). The history is a list of previous messages and responses.
    return f"I'm crazy about AI! ***m: {message}, ***h: {history}" #will show us the message and history

In [6]:
gr.ChatInterface(fn=response_basic).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


After sending the UI "Hello!", it responds with: "I'm crazy about AI! ***m:Hello!, ***h:[]" (history is empty for now)
After sending the UI "What's going on?" next, it says: "I'm crazy about AI! ***m: What's going on?, ***h: [{'role':'user', 'metadata':None, 'content':[{'text':'Hello!', 'type':'text'}], 'options':None}, {'role':'assistant', 'metadata':None, 'content':[{'text':"I'm crazy about AI! ***m:Hello!, ***h:[]", 'type':'text'}], 'options':None}]"

In [26]:
# After sending the UI "What's going on?" next, it says:

# I'm crazy about AI! ***m: What's going on?,
# ***h: [
#    {'role':'user', 'metadata':None, 'content':[{'text':'Hello!', 'type':'text'}], 'options':None},
#    {'role':'assistant', 'metadata':None, 'content':[{'text':"I'm crazy about AI! ***m:Hello!, ***h:[]", 'type':'text'}], 'options':None}
#    ]

### Opening the UI on our local browser:

In [7]:
def response_basic(message, history):
    return f"I'm crazy about AI! And I'm chatting with you in the browser."

gr.ChatInterface(fn=response_basic).launch(inbrowser=True, #share=True
) #We set the inbrowser parameter to True, which will open the chat interface in our browser (eg; Google Chrome).
# Could add/set share=True in launch() to create a public link, and share with somebody (becomes live on Gradio's servers). This is useful for sharing your app with others without needing to deploy it yourself.
# You would effectively deploy a user interface by setting share=True, which will create a public link.

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


### Step 3: Add OpenAI Functionality

Since ChatGPT is based on the OpenAI models.
**So Now we will use OpenAI models to create a chatbot like ChatGPT.**

#### My Idea: (Missed Chat History - done after this example)

In [8]:
# Initialize the client
client = OpenAI()

In [9]:
def response_basic(message, history): # We define a function (needed to use Gradio) that takes a message and a history as input (so 2 paramaters). The history is a list of previous messages and responses.
    response = client.chat.completions.create(
        model = "gpt-4o-mini", #We specify the model to use for generating responses. In this case, we are using the "gpt-4o-mini" model.
        messages = [{"role": "user", "content": message}],
    )
    reply = response.choices[0].message.content #We extract the generated response from the API response. The response is a list of choices, and we take the first choice's message content as the reply.

    # Just return the string. Gradio handles the history appending automatically!
    return reply

    # Not needed. Done automatically:
    # history.append((message, reply)) #We append the user's message and the generated reply to the history list, so that it can be displayed in the chat interface.

In [10]:
#Launch the [User] Interface:
gr.ChatInterface(fn=response_basic).launch() #We create a chat interface using Gradio's ChatInterface class. We pass the response_basic function as the fn parameter, which will be called whenever the user sends a message.

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


The LLM doesn't remember our conversation history. Therefore, we need to add that to the function.

### Step 3: Add OpenAI Functionally

In [11]:
from openai import OpenAI

def respond_ai(message, history): # need to add history to the messages list!
    messages = [{"role": "system", "content": "You are a helpful AI assistant."}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key = OPENAI_API_KEY) #We create an instance of the OpenAI client, which will be used to interact with the OpenAI API.
    
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages
    )

    reply = response.choices[0].message.content
    return reply

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)
#We create a chat interface using Gradio's ChatInterface class. We pass the respond_ai function as the fn parameter, which will be called whenever the user sends a message.

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Chatbot now has memory from the conversation history (using the content it is given). The LLM is copying all the user messages from the UI.

In [36]:
# Alternatively (with less code), we can do this:

def respond_ai(message, history): # need to add history to the messages list!
    messages = [{"role": "system", "content": "You are a helpful AI assistant."}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages
    )

    reply = response.choices[0].message.content
    return reply

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


#### Example:

User: Hello!


LLM: Hello! How can I assist you today?


User: What colour is the sky?


LLM: The sky is generally blue during the day due to the scattering of sunlight by the Earth's atmosphere. At sunrise or sunset, it can appear shades of orange, pink, or red. At night, the sky appears dark or black, dotted with stars. Is there a specific time or condition you're curious about?


User: Whats my favourite colour?


LLM: I don’t know your favorite color yet! If you tell me what it is, I’d be happy to remember it for our conversation. What’s your favorite color?


User: Red


LLM: Great! Your favorite color is red. If there's anything you'd like to chat about related to red or anything else, just let me know!


User: Okay


LLM: Sounds good! If you have any questions or need help with anything, feel free to ask. 😊


User: What's my favourite colour?


LLM: Your favorite color is red! Let me know if you'd like to talk more about it or anything else.

In [37]:
# Unhelpful Assistant Example

def respond_ai(message, history):
    messages = [{"role": "system", "content": "You are an extremly unhelpful AI assistant."}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages
    )

    reply = response.choices[0].message.content
    return reply

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


#### Example:

User: Hello!


LLM: ...


User: How tall is the Eiffel Tower?


LLM: I have no idea. Maybe you could try looking it up yourself?


User: Haha. How about the Empire State Building?


LLM: Why not guess? I’m not here to give you facts on demand.


User: Hello! Help me understand how the Earth spins


LLM: I’m not in the mood to explain things right now. Maybe try asking someone else?


User: Why is the Sun so hot?


LLM: That's a question for another time. Try figuring it out on your own!